# S2 orbit fit -- Brans-Dicke

This notebook runs the full pipeline for weak-field Brans Dicke metric with,
g_tt(r) = -(1 - 2M/r)
g_rr(r) =  1 + beta * (2M/r),     where    beta = (1+omega_bd)/(2+omega_bd).

All the physics and priors are in one file, `snope/metrics/brans_dicke.py`. Corrections can be made there.

In [ ]:
from snope.metrics.brans_dicke import main, metric, param_priors
from snope.priors import PriorMode
from snope.orbit_model import S2OrbitModel
from snope import plotting


## 1. Effective Potential

Before a full MCMC run, we check if the effective potential `V_eff(r)` actually has two turning points (periapsis and apoapsis) at some trial parameter values. If this requirement is not met, the integrator will fail.

In [ ]:
model = S2OrbitModel(metric, "../data/tab_gillessen_pos.csv", "../data/tab_gillessen_vr.csv", verbose=False)

preview = model.preview_effective_potential(
    M_bh=4.3e6, distance=8.33, a=125.5, e=0.884,
    omega_bd=10.0,
)
plotting.effective_potential_plot(preview, "Brans-Dicke")

Try a few different values of the new parameter here (and of `a`/`e`) if you're
not sure what range makes sense physically. 

## 2. Run the fit

Defaults use mixed priors: flat on the orbital elements and the new-physics
parameter (in this case, `omega_bd`), Gaussian on the offsets and `t_peri`.

In [ ]:
sampler, flat_samples, best_fit = main(
    data_path_pos="../data/tab_gillessen_pos.csv",
    data_path_rv="../data/tab_gillessen_vr.csv",
)
best_fit

## Other prior modes and settings

```python
# every parameter flat
main(prior_mode=PriorMode.FLAT)

# every parameter Gaussian
main(prior_mode=PriorMode.GAUSSIAN)

# override any pipeline setting (n_steps, n_walkers, n_points, ...)
main(prior_mode=PriorMode.MIXED, n_walkers=64, n_steps=50000, burn_in=5000)

# quick local run to sanity-check everything works, before scaling up
main(n_walkers=8, n_steps=200, burn_in=20, n_points=200)
```